# Run a managed PMLDL experiment on Kaggle

Attach the private Git bundle and official competition data. Set `MANAGED_NOTEBOOK` to the generated experiment notebook. Training may use Internet On for the pinned base model; the final inference notebook must use Internet Off.

In [ ]:
from pathlib import Path

MANAGED_NOTEBOOK = "output/jupyter-notebook/E20260905212645934620__e2-deberta-full-vs-lora-vs.ipynb"
BUNDLE_GLOB = "/kaggle/input/**/*.bundle"
WHEEL_DIR = None  # Example: /kaggle/input/pmldl-pinned-wheels
REPOSITORY_DIR = Path("/kaggle/working/PMLDL-llm-classification-finetuning")

In [ ]:
import glob
import os
import shutil
import subprocess
import sys

bundles = sorted(glob.glob(BUNDLE_GLOB, recursive=True))
if len(bundles) != 1:
    raise RuntimeError(f"Expected exactly one attached .bundle, found {bundles}")
if REPOSITORY_DIR.exists():
    shutil.rmtree(REPOSITORY_DIR)
subprocess.run(["git", "clone", bundles[0], str(REPOSITORY_DIR)], check=True)

pip_command = [sys.executable, "-m", "pip", "install"]
if WHEEL_DIR:
    pip_command += ["--no-index", "--find-links", WHEEL_DIR]
pip_command += [
    "-r", str(REPOSITORY_DIR / "requirements-baseline.lock"),
    "-r", str(REPOSITORY_DIR / "requirements-transformer.lock"),
    "--no-build-isolation", "-e", str(REPOSITORY_DIR),
]
subprocess.run(pip_command, check=True)

competition_dirs = [
    path.parent
    for path in Path("/kaggle/input").glob("**/train.csv")
    if (path.parent / "test.csv").is_file()
    and (path.parent / "sample_submission.csv").is_file()
]
if len(competition_dirs) != 1:
    raise RuntimeError(f"Expected one official competition input, found {competition_dirs}")
os.environ["PMLDL_DATA_DIR"] = str(competition_dirs[0])
print("Repository:", REPOSITORY_DIR)
print("Competition data:", competition_dirs[0])

In [ ]:
import nbformat
from nbclient import NotebookClient

source_path = REPOSITORY_DIR / MANAGED_NOTEBOOK
if not source_path.is_file():
    raise FileNotFoundError(source_path)
notebook = nbformat.read(source_path, as_version=4)
client = NotebookClient(
    notebook,
    timeout=None,
    kernel_name="python3",
    resources={"metadata": {"path": str(REPOSITORY_DIR)}},
)
client.execute()
executed_path = Path("/kaggle/working/executed-managed-experiment.ipynb")
nbformat.write(notebook, executed_path)
print("Executed notebook:", executed_path)
print("Download results from:", REPOSITORY_DIR / "results")
print("Download artifacts from:", REPOSITORY_DIR / "artifacts")